# Example 6.5 / 1: Two Problems over One Model

Companion notebook for **6.5 Several Models in One Session** in the AMPL lab Documentation, section "Two Problems over One Model". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Two Problems over One Model

The steel model below has two problems. `Plan` maximizes profit within the available hours.
`Lean` minimizes the hours used while earning at least `target`, and the script sets `target`
to 90% of the profit `Plan` found. `ampl.eval()` sends the `problem` command, which has no amplpy
method:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                    # products
param rate {PROD} > 0;       # tons produced per hour
param profit {PROD};         # profit per ton
param market {PROD} >= 0;   # most that can be sold, in tons
param avail >= 0;            # hours available
param target default 0;      # profit Lean must reach, set by the script
var Make {p in PROD} >= 0, <= market[p];  # tons produced

maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];
minimize Hours: sum {p in PROD} (1 / rate[p]) * Make[p];
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;
subject to Profit_Floor: sum {p in PROD} profit[p] * Make[p] >= target;

problem Plan: Make, Total_Profit, Time;              # best profit within the hours
problem Lean: Make, Hours, Time, Profit_Floor;       # fewest hours for the target profit

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.eval("problem Plan;")
ampl.solve(solver="highs", verbose=False)
best = ampl.obj["Total_Profit"].value()
ampl.param["target"] = 0.9 * best

ampl.eval("problem Lean;")
ampl.solve(solver="highs", verbose=False)
print("Plan profit:", best)
print("Lean hours:", round(ampl.obj["Hours"].value(), 2), ampl.var["Make"].to_dict())

Plan profit: 192000.0
Lean hours: 35.43 {'bands': 6000, 'coils': 760}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param:    rate  profit  market :=
  bands    200      25    6000
  coils    140      30    4000;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
problem Plan;
option solver highs, solver_msg 0;
solve;
let target := 0.9 * Total_Profit;
problem Lean;
option solver highs, solver_msg 0;
solve;
display Total_Profit, Hours, Make;

Total_Profit = 172800
Hours = 35.4286

Make [*] :=
bands  6000
coils   760
;

